# 00. Colab 환경 셋업 점검

**이 노트북이 하는 일:** TeachArm ACT 학습 파이프라인을 Colab에서 돌리기 전에, 실행 환경이 준비되었는지 처음부터 끝까지 한 번에 확인합니다.

1. Drive 연결, Colab Secrets에서 토큰 불러오기
2. GitHub 저장소 받기
3. 공통 셋업 `code/training/colab_setup.sh` 실행 (패키지 설치, Drive 폴더 생성, 실행 환경 기록)
4. 설치된 PyTorch와 GPU 확인
5. Hugging Face, W&B 로그인 확인
6. PoC 데이터셋 `lerobot/svla_so100_pickplace` 정보 확인
7. 카메라 영상 프레임 디코딩 확인
8. 학습 명령 `lerobot-train` 확인

| 항목 | 내용 |
| --- | --- |
| 언제 실행 | 최초 1회, 그리고 `requirements-train.txt`를 바꾼 뒤 |
| 런타임 | CPU로 충분 (GPU 런타임이면 GPU 정보도 함께 확인) |
| 사전 준비 | Colab Secrets에 `HF_TOKEN`, `WANDB_API_KEY` 등록, 이 노트북의 액세스 허용 |
| 결과 | 마지막 셀에 모든 항목 통과 표시, 환경 기록은 `MyDrive/teacharm/env/` |

01~03번 노트북도 1~3번 셀은 똑같이 실행합니다. 위에서부터 차례로 실행하세요.

## 1. Drive 연결, 토큰 불러오기

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")

# 토큰은 환경 변수로만 둡니다. huggingface_hub와 wandb가 이 변수를 자동으로 읽습니다.
for name in ("HF_TOKEN", "WANDB_API_KEY"):
    os.environ[name] = userdata.get(name)

passed = ["Drive 연결, Secrets 불러오기"]
print("OK")

## 2. 저장소 받기

In [ ]:
REPO_URL = "https://github.com/jen282/physical-ai.git"
REPO_DIR = "/content/physical-ai"
BRANCH = "main"

!if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} pull --ff-only; else git clone -b {BRANCH} {REPO_URL} {REPO_DIR}; fi
!git -C {REPO_DIR} log -1 --oneline
passed.append("저장소 받기")

## 3. 공통 셋업

처음 실행하면 패키지 설치에 몇 분 걸립니다. 출력 중 `⚠`가 보이면 안내에 따라 세션을 다시 시작하세요.

In [ ]:
!bash {REPO_DIR}/code/training/colab_setup.sh
passed.append("공통 셋업")

## 4. PyTorch, GPU 확인

In [ ]:
import importlib.metadata as md
import torch

# 셋업 전에 torch를 import한 상태라면 메모리의 버전과 설치된 버전이 다를 수 있습니다.
installed = md.version("torch")
assert torch.__version__.split("+")[0] == installed.split("+")[0], (
    f"메모리의 torch({torch.__version__})와 설치된 torch({installed})가 다릅니다. "
    "'런타임 → 세션 다시 시작' 후 1번 셀부터 다시 실행하세요."
)
print(f"torch {torch.__version__}, lerobot {md.version('lerobot')}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    x = torch.randn(1024, 1024, device="cuda")
    torch.cuda.synchronize()
    print(f"GPU {props.name}, 메모리 {props.total_memory / 1e9:.1f} GB, CUDA {torch.version.cuda}, 연산 확인 {(x @ x).shape}")
else:
    print("CPU 런타임입니다. 학습(02번)에는 '런타임 → 런타임 유형 변경'에서 T4/L4 GPU를 고르세요.")
passed.append("PyTorch, GPU")

## 5. Hugging Face, W&B 로그인 확인

In [ ]:
import wandb
from huggingface_hub import whoami

me = whoami()
role = me.get("auth", {}).get("accessToken", {}).get("role")
print(f"Hugging Face: {me['name']} (토큰 권한: {role})")
if role == "read":
    print("⚠ read 토큰입니다. 체크포인트를 Hub에 올리려면 write 토큰이 필요합니다.")

assert wandb.login(key=os.environ["WANDB_API_KEY"], verify=True), "W&B API 키를 확인하세요."
print("W&B: 로그인 확인")
passed.append("Hugging Face, W&B 로그인")

## 6. 데이터셋 정보 확인

데이터셋은 커밋을 고정해서 받습니다(`v3.0` 태그와 같은 커밋). 메타데이터만 받으므로 빠릅니다.

In [ ]:
from lerobot.datasets import LeRobotDatasetMetadata

REPO_ID = "lerobot/svla_so100_pickplace"
REVISION = "728583b5eaf9e739a7f119e2def466fa1d552402"

meta = LeRobotDatasetMetadata(REPO_ID, revision=REVISION)
expected = {
    "에피소드 수": 50,
    "fps": 30,
    "카메라": ["observation.images.top", "observation.images.wrist"],
    "관절 상태 차원": 6,
    "행동 차원": 6,
}
actual = {
    "에피소드 수": meta.total_episodes,
    "fps": meta.fps,
    "카메라": sorted(meta.camera_keys),
    "관절 상태 차원": meta.features["observation.state"]["shape"][0],
    "행동 차원": meta.features["action"]["shape"][0],
}
for key, want in expected.items():
    print(f"{'OK ' if actual[key] == want else 'X  '} {key}: {actual[key]} (기대값 {want})")
assert actual == expected, "데이터셋 구성이 예상과 다릅니다."
print(f"총 프레임 {meta.total_frames}, 로봇 {meta.robot_type}")
passed.append("데이터셋 정보")

## 7. 영상 프레임 디코딩 확인

에피소드 0번의 첫 프레임을 카메라별로 띄웁니다. 영상 파일을 처음 받을 때 시간이 조금 걸립니다.

In [ ]:
import matplotlib.pyplot as plt
from lerobot.datasets import LeRobotDataset

ds = LeRobotDataset(REPO_ID, revision=REVISION, episodes=[0])
frame = ds[0]

fig, axes = plt.subplots(1, len(meta.camera_keys), figsize=(10, 4))
for ax, key in zip(axes, meta.camera_keys):
    ax.imshow(frame[key].permute(1, 2, 0).numpy())  # (C, H, W) 0~1 → (H, W, C)
    ax.set_title(f"{key} {tuple(frame[key].shape)}")
    ax.axis("off")
plt.show()

print("관절 상태", frame["observation.state"].numpy().round(2))
print("행동    ", frame["action"].numpy().round(2))
print("작업    ", frame["task"])
passed.append("영상 디코딩")

## 8. 학습 명령 확인

In [ ]:
import subprocess

out = subprocess.run(["lerobot-train", "--help"], capture_output=True, text=True)
assert out.returncode == 0, out.stderr[-2000:]

# 02번 노트북에서 쓸 옵션이 도움말에 있는지 확인합니다.
for flag in ("--policy.type", "--dataset.repo_id", "--dataset.episodes", "--steps",
             "--save_freq", "--output_dir", "--resume", "--wandb.enable"):
    print(f"{'OK ' if flag in out.stdout else '?  '} {flag}")
passed.append("lerobot-train")

## 결과

In [ ]:
for item in passed:
    print(f"✓ {item}")
print(f"\n{len(passed)}/8 항목 통과")
!ls -t /content/drive/MyDrive/teacharm/env | head -1